<a href="https://colab.research.google.com/github/Darya-06/python-ai-Aleinik-Darya/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- `animated_television_series.csv` — информация об анимационных телесериалах (название, страна, композитор, рейтинг и производные работы)

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем CSV-файл в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [1]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-Aleinik-Darya"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Aleinik-Darya
    !git clone -q https://github.com/Darya-06/python-ai-Aleinik-Darya.git

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-Aleinik-Darya
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Aleinik-Darya


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [ ]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_series = pd.read_csv("data/animated_television_series.csv")


print("✅ Загружено строк в df_series:", len(df_series))


✅ Загружено строк в df_series: 4557


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `serial` с URL Wikidata (например, `http://www.wikidata.org/entity/Q77522`) — нам не нужна ссылка, достаточно читаемого названия сериала.
- Столбцы `serialLabel`, `countryLabel`, `composerLabel`, `rarsRatingLabel`, `derivativeWorkLabel` содержат человекочитаемые значения.

В этом шаге мы:
- удалим столбец `serial` с техническими URL;
- переименуем все столбцы с суффиксом `Label`, убрав его (`serialLabel → serial`, `countryLabel → country` и т.д.);
- получим чистую таблицу для дальнейшего анализа.

> ⚠️ **Важно:** столбец `rarsRating` может содержать текстовые значения (например, "18+"), поэтому числовое преобразование здесь не применяется.

In [ ]:
# 🧹 Шаг 2B. Очистка и переименование столбцов (безопасно для повторного запуска)

# 1) Удаляем технический столбец с URL Wikidata, если он существует
if "serial" in df_series.columns and "serialLabel" in df_series.columns:
    # Удаляем ТОЛЬКО если есть и URL-столбец, и его человекочитаемая версия
    # (защита от случайного удаления уже очищенных данных)
    df_series = df_series.drop(columns=["serial"])
    print("🗑️ Удалён столбец 'serial' с URL Wikidata")
elif "serial" in df_series.columns and "serialLabel" not in df_series.columns:
    print("⚠️ Столбец 'serial' присутствует без 'serialLabel' — возможно, данные уже очищены")
else:
    print("ℹ️ Столбец 'serial' с URL уже удалён")

# 2) Автоматически убираем суффикс 'Label' у всех столбцов, которые его содержат
columns_to_rename = {
    col: col.replace("Label", "")
    for col in df_series.columns
    if col.endswith("Label")
}

if columns_to_rename:
    df_series = df_series.rename(columns=columns_to_rename)
    print(f"✏️ Убран суффикс 'Label' у столбцов: {list(columns_to_rename.keys())} → {list(columns_to_rename.values())}")
else:
    print("ℹ️ Суффикс 'Label' уже удалён из всех столбцов")

# 3) Итоговый вывод
print("\n✅ Данные очищены и готовы к анализу")
print("📋 Итоговая структура столбцов:", list(df_series.columns))
print("\n🔍 Пример данных после очистки:")
display(df_series.head(3))

⚠️ Столбец 'serial' присутствует без 'serialLabel' — возможно, данные уже очищены
ℹ️ Суффикс 'Label' уже удалён из всех столбцов

✅ Данные очищены и готовы к анализу
📋 Итоговая структура столбцов: ['serial', 'country', 'composer', 'rarsRating', 'derivativeWork']

🔍 Пример данных после очистки:


,serial,country,composer,rarsRating,derivativeWork
0,Клуб Винкс,Италия,Фабрицио Кастаниа,NaN,Клуб Винкс: Волшебное приключение
1,Клуб Винкс,Италия,Фабрицио Кастаниа,NaN,Судьба: Сага Винкс
2,Футурама,США,Кристофер Тинг,18+,NaN


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор обоих DataFrame:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по бюджету (`capital_cost`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код два раза.

In [ ]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов, пропуски и первые строки."""
    print(f"\n📊 {name}")
    print(f"Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")
    print("Столбцы:", ", ".join(df.columns))

    # Анализ пропусков
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("\nПропуски в данных:")
        for col, count in missing[missing > 0].items():
            pct = count / len(df) * 100
            print(f"  • {col}: {count} ({pct:.1f}%)")
    else:
        print("\n✅ Пропусков в данных нет")

    print("\nПервые строки:")
    display(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_series, "Анимационные телесериалы (df_series)")

# Дополнительная статистика по ключевым категориям
print("\n📈 Уникальные значения:")
print(f"  • Сериалов: {df_series['serial'].nunique()}")
print(f"  • Стран: {df_series['country'].nunique()} ({', '.join(df_series['country'].unique()[:5])}...)")
print(f"  • Композиторов: {df_series['composer'].nunique()}")

if 'rarsRating' in df_series.columns:
    ratings = df_series['rarsRating'].value_counts(dropna=False)
    print(f"\n📺 Рейтинги RARS:")
    for rating, count in ratings.items():
        label = "Без рейтинга" if pd.isna(rating) else rating
        print(f"  • {label}: {count}")


📊 Анимационные телесериалы (df_series)
Размер: 4557 строк × 5 столбцов
Столбцы: serial, country, composer, rarsRating, derivativeWork

Пропуски в данных:
  • composer: 3344 (73.4%)
  • rarsRating: 4414 (96.9%)
  • derivativeWork: 4253 (93.3%)

Первые строки:


,serial,country,composer,rarsRating,derivativeWork
0,Клуб Винкс,Италия,Фабрицио Кастаниа,NaN,Клуб Винкс: Волшебное приключение
1,Клуб Винкс,Италия,Фабрицио Кастаниа,NaN,Судьба: Сага Винкс
2,Футурама,США,Кристофер Тинг,18+,NaN
3,Время приключений,США,NaN,NaN,Время приключений: Дальние земли
4,Клуб Винкс,США,Питер Зиззо,NaN,Клуб Винкс: Тайна затерянного королевства



📈 Уникальные значения:
  • Сериалов: 3461
  • Стран: 98 (Италия, США, Великобритания, Россия, Канада...)
  • Композиторов: 462

📺 Рейтинги RARS:
  • Без рейтинга: 4414
  • 0+: 54
  • 6+: 42
  • 12+: 29
  • 18+: 14
  • 16+: 4


## 🔬 [4] Глубокая валидация: мультинациональные сериалы и композиторы-«монополисты»

В этом анализе мы выйдем за рамки базовой статистики и исследуем две уникальные особенности индустрии анимации:

#### 🌍 Мультинациональные сериалы
Сериалы часто создаются при участии нескольких стран (совместное производство). В нашем датасете каждая строка = одна страна участия, поэтому сериалы с 2+ странами представлены несколькими записями. Мы выявим:
- Сериалы с наибольшим числом стран-соавторов
- Неожидан…ония + Бразилия)
- Долю «глобальных» проектов от общего числа сериалов

#### 🎼 Композиторы-«монополисты»
Некоторые сериалы на протяжении всей франшизы (включая спин-оффы и адаптации) сохраняют одного композитора — это признак сильной авторской концепции звукового оформления. Мы определим:
- Сериалы, где все версии имеют единого композитора
- Топ-5 композиторов-«монополистов» по числу таких сериалов
- Контраст: сериалы с частой сменой композиторов (признак перезапусков или кризисов производства)

In [ ]:
# 🔬 Шаг 4. Глубокая валидация данных

print("=" * 70)
print("🌍 АНАЛИЗ 1: МУЛЬТИНАЦИОНАЛЬНЫЕ СЕРИАЛЫ (совместное производство)")
print("=" * 70)

# Считаем уникальные страны для каждого сериала
country_counts = df_series.groupby('serial')['country'].nunique().sort_values(ascending=False)

# Фильтруем сериалы с 2+ странами
multi_country_series = country_counts[country_counts >= 2]
total_series = df_series['serial'].nunique()
multi_country_pct = (len(multi_country_series) / total_series) * 100

print(f"\n📈 Статистика:")
print(f"  • Всего уникальных сериалов: {total_series}")
print(f"  • Сериалов с 2+ странами: {len(multi_country_series)} ({multi_country_pct:.1f}%)")
print(f"  • Максимум стран у одного сериала: {multi_country_series.max()}")

# Детали для топ-10 мультинациональных сериалов
print("\n🏆 Топ-10 сериалов по числу стран-соавторов:")
top_multi = multi_country_series.head(10).reset_index()
top_multi.columns = ['serial', 'country_count']

# Добавляем список стран для каждого сериала
def get_countries(serial_name):
    countries = df_series[df_series['serial'] == serial_name]['country'].unique()
    return ', '.join(sorted(countries))

top_multi['countries'] = top_multi['serial'].apply(get_countries)
display(top_multi[['serial', 'country_count', 'countries']].style.set_properties(**{'text-align': 'left'}))

# Анализ самых частых пар стран
print("\n🤝 Самые частые пары стран в совместных проектах:")
country_pairs = []
for serial in multi_country_series.index[:50]:  # берём топ-50 для анализа
    countries = sorted(df_series[df_series['serial'] == serial]['country'].unique())
    if len(countries) >= 2:
        for i in range(len(countries)):
            for j in range(i+1, len(countries)):
                pair = tuple(sorted([countries[i], countries[j]]))
                country_pairs.append(pair)

from collections import Counter
pair_counts = Counter(country_pairs).most_common(5)
for pair, count in pair_counts:
    print(f"  • {pair[0]} + {pair[1]}: {count} совместных проектов")

print("\n" + "=" * 70)
print("🎼 АНАЛИЗ 2: КОМПОЗИТОРЫ-«МОНОПОЛИСТЫ»")
print("=" * 70)

# Проверяем уникальность композитора для каждого сериала
composer_uniqueness = df_series.groupby('serial')['composer'].nunique()
monopolist_series = composer_uniqueness[composer_uniqueness == 1].index.tolist()
non_monopolist_series = composer_uniqueness[composer_uniqueness > 1].index.tolist()

monopolist_pct = (len(monopolist_series) / total_series) * 100

print(f"\n📈 Статистика:")
print(f"  • Сериалов с единым композитором: {len(monopolist_series)} ({monopolist_pct:.1f}%)")
print(f"  • Сериалов со сменой композитора: {len(non_monopolist_series)} ({100 - monopolist_pct:.1f}%)")

# Топ-5 композиторов-монополистов по числу сериалов
print("\n🏆 Топ-5 композиторов-«монополистов» (сервис единого звучания):")
monopolist_composers = df_series[df_series['serial'].isin(monopolist_series)]
top_composers = monopolist_composers.groupby('composer')['serial'].nunique().sort_values(ascending=False).head(5)

for rank, (composer, count) in enumerate(top_composers.items(), 1):
    # Примеры сериалов для этого композитора
    examples = monopolist_composers[monopolist_composers['composer'] == composer]['serial'].unique()[:3]
    examples_str = ', '.join(examples)
    print(f"  {rank}. {composer} — {count} сериалов")
    print(f"     Примеры: {examples_str}")

# Контраст: сериалы с максимальной сменой композиторов
print("\n⚠️  Сериалы с наибольшей сменой композиторов (признак перезапусков):")
composer_changes = composer_uniqueness[composer_uniqueness > 1].sort_values(ascending=False).head(5)
for serial, count in composer_changes.items():
    composers = ', '.join(df_series[df_series['serial'] == serial]['composer'].unique())
    print(f"  • {serial}: {count} композитора")
    print(f"    → {composers}")

print("\n✅ Анализ завершён. Данные готовы к углублённому исследованию.")

🌍 АНАЛИЗ 1: МУЛЬТИНАЦИОНАЛЬНЫЕ СЕРИАЛЫ (совместное производство)

📈 Статистика:
  • Всего уникальных сериалов: 3461
  • Сериалов с 2+ странами: 547 (15.8%)
  • Максимум стран у одного сериала: 8

🏆 Топ-10 сериалов по числу стран-соавторов:


,serial,country_count,countries
0,Говорящий Том и друзья,8,"Австрия, Великобритания, Испания, Кипр, США, Сингапур, Словения, Таиланд"
1,Talking Tom & Friends Minis,5,"Великобритания, Кипр, США, Словения, Япония"
2,Q118955021,5,"Аргентина, Испания, Колумбия, Португалия, Чили"
3,Агент 203,5,"Германия, Индия, Италия, США, Северная Македония"
4,Зак Шторм,5,"Индонезия, Италия, Республика Корея, США, Франция"
5,Billy the Cat,5,"Бельгия, Великобритания, Германия, Канада, Франция"
6,The Nordic Christmas Hour,5,"Дания, Исландия, Норвегия, Финляндия, Швеция"
7,Книга джунглей,5,"Великобритания, Германия, Индия, Финляндия, Франция"
8,Викинг Вик,5,"Австралия, Австрия, Германия, Нидерланды, Франция"
9,Ferdy the Ant,5,"Великобритания, Германия, Китайская Республика (Тайвань), Лихтенштейн, Чехословакия"



🤝 Самые частые пары стран в совместных проектах:
  • Канада + США: 11 совместных проектов
  • Канада + Франция: 10 совместных проектов
  • Великобритания + США: 9 совместных проектов
  • Великобритания + Франция: 9 совместных проектов
  • США + Франция: 8 совместных проектов

🎼 АНАЛИЗ 2: КОМПОЗИТОРЫ-«МОНОПОЛИСТЫ»

📈 Статистика:
  • Сериалов с единым композитором: 650 (18.8%)
  • Сериалов со сменой композитора: 67 (81.2%)

🏆 Топ-5 композиторов-«монополистов» (сервис единого звучания):
  1. Хойт Куртин — 55 сериалов
     Примеры: Yogi's Treasure Hunt, Пёс Хакльберри, The Magilla Gorilla Show
  2. Майкл Тавера — 22 сериалов
     Примеры: Time Squad, ¡Mucha Lucha!, Лило и Стич
  3. Шуки Леви — 15 сериалов
     Примеры: Непобедимая принцесса Ши-Ра, Настоящие охотники за привидениями, Инспектор Гаджет
  4. Гай Мун — 11 сериалов
     Примеры: Ужасные приключения Билли и Мэнди, Волшебные покровители, Дэнни-призрак
  5. Petr Skoumal — 11 сериалов
     Примеры: Боб и Бобек, Q12048159, O zvířátk

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали ваш репозиторий `Darya-06/python-ai-Aleinik-Darya` в Google Colab
- ✅ Прочитали CSV-файл `data/animated_television_series.csv` с данными об анимационных сериалах из Викиданных
- ✅ Очистили данные:
  - удалили технический столбец `serial` с URL Wikidata
  - автоматически убрали суффикс `Label` у всех столбцов (`serialLabel → serial`, `countryLabel → country` и т.д.)
- ✅ Провели базовый обзор:
  - 4 557 записей, 5 информативных столбцов
  - 3 461 уникальный сериал, 98 стран, 462 композитора
  - анализ пропусков (97% записей без рейтинга RARS)
- ✅ Выполнили глубокую валидацию:
  - выявили мультинациональные сериалы (совместное производство 2+ стран)
  - нашли композиторов-«монополистов» — авторов, сохраняющих единое звучание на протяжении всей франшизы
  - проанализировали распределение возрастных рейтингов (от 0+ до 18+)

Теперь у нас есть **чистый, структурированный датасет** об анимационных сериалах, готовый к углублённому анализу.

В ноутбуке следующей недели мы будем использовать **этот же датасет** для:
- построения визуализаций (распределение стран, эволюция рейтингов по годам)
- анализа культурных паттернов (какие страны предпочитают «взрослый» контент)
- исследования связей между сериалами и их производными работами. 🌐🎬